# Brain Tumor DX - Evaluation Only

Load pre-trained checkpoints and evaluate on train/test sets.

**Pipeline:**
1. Detect GPU + configure device
2. Install dependencies
3. Setup classification data
4. Load classifier + evaluate on train/test
5. **Comprehensive classifier evaluation** (accuracy, precision, recall, F1, AUC-ROC, confusion matrix)
6. Setup segmentation data (BraTS)
7. Load segmenter + evaluate on train/test
8. **Comprehensive segmenter evaluation** (Dice, IoU, Hausdorff distance, volume similarity)

In [ ]:
#@title Cell 1: Detect GPU + configure device
import os, sys, torch

# Find project root: walk up from CWD looking for pyproject.toml
_cwd = os.path.abspath('.')
_proj = _cwd
while _proj != os.path.dirname(_proj):
    if os.path.isfile(os.path.join(_proj, 'pyproject.toml')):
        break
    _proj = os.path.dirname(_proj)
else:
    _parent = os.path.dirname(_cwd)
    if os.path.isfile(os.path.join(_parent, 'pyproject.toml')):
        _proj = _parent
    else:
        _proj = _cwd

PROJECT_DIR = _proj
os.chdir(PROJECT_DIR)

DATA_DIR = os.path.join(PROJECT_DIR, 'data')
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.environ['DEVICE'] = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE = os.environ['DEVICE']

src_path = os.path.join(PROJECT_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA compiled: {torch.version.cuda}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM: {props.total_memory / 1e9:.1f} GB')
    print(f'Compute capability: {props.major}.{props.minor}')
else:
    print('WARNING: No GPU. Training will be very slow.')

print(f'\nProject root: {PROJECT_DIR}')
print(f'Data: {DATA_DIR}')
print(f'Device: {DEVICE}')

from brain_tumor_dx.config import settings
settings.device = DEVICE
print(f'Classes: {settings.tumor_classes}')

In [ ]:
#@title Cell 2: Install dependencies
!pip install -q torch torchvision monai nibabel pydicom scikit-image tqdm pydantic python-dotenv scikit-learn matplotlib seaborn
print('Done.')

In [ ]:
#@title Cell 3: Setup classification data
import os

CLASS_DATA = os.path.join(DATA_DIR, 'raw', 'classification')

if not os.path.isdir(CLASS_DATA):
    for candidate in [DATA_DIR, os.path.join(DATA_DIR, 'raw')]:
        if os.path.isdir(os.path.join(candidate, 'glioma')):
            CLASS_DATA = candidate
            break
        for sub in ['Training', 'Testing']:
            path = os.path.join(candidate, sub)
            if os.path.isdir(os.path.join(path, 'glioma')):
                CLASS_DATA = path
                break

if os.path.isdir(CLASS_DATA):
    rename_map = {'notumor': 'no_tumor', 'Notumor': 'no_tumor', 'NO_TUMOR': 'no_tumor'}
    for old, new in rename_map.items():
        old_path = os.path.join(CLASS_DATA, old)
        new_path = os.path.join(CLASS_DATA, new)
        if os.path.isdir(old_path) and not os.path.isdir(new_path):
            os.rename(old_path, new_path)
            print(f'Renamed: {old} -> {new}')

    for sub in ['Training', 'Testing']:
        src = os.path.join(CLASS_DATA, sub)
        if os.path.isdir(src):
            for item in os.listdir(src):
                s = os.path.join(src, item)
                d = os.path.join(CLASS_DATA, item)
                if os.path.isdir(s) and not os.path.exists(d):
                    os.rename(s, d)
            os.rmdir(src)

    print(f'Classification data: {CLASS_DATA}')
    total = 0
    for cls in settings.tumor_classes:
        cls_path = os.path.join(CLASS_DATA, cls)
        if os.path.isdir(cls_path):
            n = len([f for f in os.listdir(cls_path) if os.path.isfile(os.path.join(cls_path, f))])
            print(f'  {cls}: {n} images')
            total += n
        else:
            print(f'  {cls}: MISSING')
    print(f'  Total: {total} images')
else:
    print(f'Classification data not found at {CLASS_DATA}')
    print('Expected layout: data/raw/classification/{glioma,meningioma,pituitary,no_tumor}/*.jpg')

In [ ]:
#@title Cell 4: Load Classifier + Evaluate on Train/Test
import os, sys, torch
import numpy as np
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm

def _find_root(start):
    d = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(d, 'src', 'brain_tumor_dx')):
            return d
        parent = os.path.dirname(d)
        if parent == d: break
        d = parent
    return start

PROJECT_DIR = _find_root(os.getcwd())
os.chdir(PROJECT_DIR)
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
CLASS_DATA = os.path.join(DATA_DIR, 'raw', 'classification')
if not os.path.isdir(CLASS_DATA):
    for c in [DATA_DIR, os.path.join(DATA_DIR, 'raw')]:
        if os.path.isdir(os.path.join(c, 'glioma')):
            CLASS_DATA = c
            break

src_path = os.path.join(PROJECT_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

DEVICE = os.environ.get('DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')
os.environ['DEVICE'] = DEVICE

from brain_tumor_dx.config import settings
settings.device = DEVICE
from brain_tumor_dx.data.datasets import ClassificationDataset
from brain_tumor_dx.models.classifier import TumorClassifier

BATCH_SIZE = 32
VAL_SPLIT = 0.15

full_dataset = ClassificationDataset(CLASS_DATA)
n_val = int(len(full_dataset) * VAL_SPLIT)
n_train = len(full_dataset) - n_val

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_dataset, [n_train, n_val], generator=generator)

print(f'Total: {len(full_dataset)} samples')
print(f'Train: {n_train} samples')
print(f'Val/Test: {n_val} samples')

CKPT = os.path.join(CHECKPOINT_DIR, 'classifier.pt')
if not os.path.exists(CKPT):
    raise FileNotFoundError(f'Checkpoint not found: {CKPT}')

model = TumorClassifier(num_classes=len(settings.tumor_classes)).to(DEVICE)
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()
print(f'Loaded checkpoint: {CKPT}')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

def evaluate_model(model, loader, device):
    all_preds = []
    all_labels = []
    all_probs = []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Evaluating', leave=False):
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

print('Evaluating on TRAIN set...')
train_preds, train_labels, train_probs = evaluate_model(model, train_loader, DEVICE)

print('Evaluating on TEST/VAL set...')
val_preds, val_labels, val_probs = evaluate_model(model, val_loader, DEVICE)

print(f'\nTrain: {len(train_preds)} samples | Test: {len(val_preds)} samples')

In [ ]:
#@title Cell 5: Comprehensive CLASSIFIER Evaluation (Train vs Test)
import os, sys, torch, numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    cohen_kappa_score, matthews_corrcoef
)
from sklearn.preprocessing import label_binarize

classes = list(settings.tumor_classes)
n_classes = len(classes)

print('=' * 70)
print('       TRAINING SET METRICS')
print('=' * 70)

train_acc = accuracy_score(train_labels, train_preds)
train_prec = precision_score(train_labels, train_preds, average='macro')
train_rec = recall_score(train_labels, train_preds, average='macro')
train_f1 = f1_score(train_labels, train_preds, average='macro')
train_kappa = cohen_kappa_score(train_labels, train_preds)
train_mcc = matthews_corrcoef(train_labels, train_preds)

train_labels_bin = label_binarize(train_labels, classes=range(n_classes))
train_auc_per_class = []
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(train_labels_bin[:, i], train_probs[:, i])
    train_auc_per_class.append(auc(fpr, tpr))
train_auc_macro = np.mean(train_auc_per_class)

print(f'  Accuracy:         {train_acc:.4f} ({train_acc*100:.2f}%)')
print(f'  Precision:        {train_prec:.4f}')
print(f'  Recall:           {train_rec:.4f}')
print(f'  F1:               {train_f1:.4f}')
print(f'  AUC-ROC:          {train_auc_macro:.4f}')
print(f'  Kappa:            {train_kappa:.4f}')
print(f'  MCC:              {train_mcc:.4f}')

print('\n' + '=' * 70)
print('       TEST/VALIDATION SET METRICS')
print('=' * 70)

test_acc = accuracy_score(val_labels, val_preds)
test_prec = precision_score(val_labels, val_preds, average='macro')
test_rec = recall_score(val_labels, val_preds, average='macro')
test_f1 = f1_score(val_labels, val_preds, average='macro')
test_kappa = cohen_kappa_score(val_labels, val_preds)
test_mcc = matthews_corrcoef(val_labels, val_preds)

test_labels_bin = label_binarize(val_labels, classes=range(n_classes))
test_auc_per_class = []
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(test_labels_bin[:, i], val_probs[:, i])
    test_auc_per_class.append(auc(fpr, tpr))
test_auc_macro = np.mean(test_auc_per_class)

print(f'  Accuracy:         {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'  Precision:        {test_prec:.4f}')
print(f'  Recall:           {test_rec:.4f}')
print(f'  F1:               {test_f1:.4f}')
print(f'  AUC-ROC:          {test_auc_macro:.4f}')
print(f'  Kappa:            {test_kappa:.4f}')
print(f'  MCC:              {test_mcc:.4f}')

colors = ['#2196F3', '#FF9800', '#4CAF50', '#F44336']

fig = plt.figure(figsize=(20, 18))
gs = gridspec.GridSpec(4, 3, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC', 'Kappa', 'MCC']
train_metrics = [train_acc, train_prec, train_rec, train_f1, train_auc_macro, train_kappa, train_mcc]
test_metrics = [test_acc, test_prec, test_rec, test_f1, test_auc_macro, test_kappa, test_mcc]
x = np.arange(len(metrics_names))
width = 0.35
bars1 = ax1.bar(x - width/2, train_metrics, width, label='Train', color='#2196F3', edgecolor='black', alpha=0.8)
bars2 = ax1.bar(x + width/2, test_metrics, width, label='Test', color='#FF9800', edgecolor='black', alpha=0.8)
ax1.set_title('Train vs Test Metrics', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics_names, rotation=45, ha='right')
ax1.set_ylabel('Score')
ax1.set_ylim(0, 1.1)
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')
for bar in bars1:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., h + 0.02, f'{h:.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., h + 0.02, f'{h:.3f}', ha='center', va='bottom', fontsize=8)

ax2 = fig.add_subplot(gs[0, 1])
cm_train = confusion_matrix(train_labels, train_preds)
im2 = ax2.imshow(cm_train, interpolation='nearest', cmap='Blues')
ax2.set_title('Confusion Matrix (Train)', fontsize=12, fontweight='bold')
ax2.set_xticks(range(n_classes))
ax2.set_yticks(range(n_classes))
ax2.set_xticklabels(classes, rotation=45, ha='right')
ax2.set_yticklabels(classes)
ax2.set_xlabel('Predicted')
ax2.set_ylabel('True')
for i in range(n_classes):
    for j in range(n_classes):
        ax2.text(j, i, str(cm_train[i, j]), ha='center', va='center',
                color='white' if cm_train[i, j] > cm_train.max()/2 else 'black', fontsize=9)
plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

ax3 = fig.add_subplot(gs[0, 2])
cm_test = confusion_matrix(val_labels, val_preds)
im3 = ax3.imshow(cm_test, interpolation='nearest', cmap='Oranges')
ax3.set_title('Confusion Matrix (Test)', fontsize=12, fontweight='bold')
ax3.set_xticks(range(n_classes))
ax3.set_yticks(range(n_classes))
ax3.set_xticklabels(classes, rotation=45, ha='right')
ax3.set_yticklabels(classes)
ax3.set_xlabel('Predicted')
ax3.set_ylabel('True')
for i in range(n_classes):
    for j in range(n_classes):
        ax3.text(j, i, str(cm_test[i, j]), ha='center', va='center',
                color='white' if cm_test[i, j] > cm_test.max()/2 else 'black', fontsize=9)
plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)

ax4 = fig.add_subplot(gs[1, 0])
per_class_prec_train = precision_score(train_labels, train_preds, average=None)
per_class_prec_test = precision_score(val_labels, val_preds, average=None)
x = np.arange(n_classes)
bars1 = ax4.bar(x - width/2, per_class_prec_train, width, label='Train', color='#2196F3', edgecolor='black', alpha=0.8)
bars2 = ax4.bar(x + width/2, per_class_prec_test, width, label='Test', color='#FF9800', edgecolor='black', alpha=0.8)
ax4.set_title('Per-Class Precision', fontsize=12, fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(classes, rotation=45, ha='right')
ax4.set_ylabel('Precision')
ax4.set_ylim(0, 1.1)
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

ax5 = fig.add_subplot(gs[1, 1])
for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(train_labels_bin[:, i], train_probs[:, i])
    ax5.plot(fpr, tpr, color=colors[i], lw=2, label=f'{cls} ({train_auc_per_class[i]:.3f})')
ax5.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax5.set_title('ROC Curves (Train)', fontsize=12, fontweight='bold')
ax5.set_xlabel('False Positive Rate')
ax5.set_ylabel('True Positive Rate')
ax5.legend(loc='lower right', fontsize=9)
ax5.grid(True, alpha=0.3)

ax6 = fig.add_subplot(gs[1, 2])
for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(test_labels_bin[:, i], val_probs[:, i])
    ax6.plot(fpr, tpr, color=colors[i], lw=2, label=f'{cls} ({test_auc_per_class[i]:.3f})')
ax6.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax6.set_title('ROC Curves (Test)', fontsize=12, fontweight='bold')
ax6.set_xlabel('False Positive Rate')
ax6.set_ylabel('True Positive Rate')
ax6.legend(loc='lower right', fontsize=9)
ax6.grid(True, alpha=0.3)

ax7 = fig.add_subplot(gs[2, 0])
max_probs_train = train_probs.max(axis=1)
max_probs_test = val_probs.max(axis=1)
ax7.hist(max_probs_train, bins=30, alpha=0.6, label=f'Train (n={len(max_probs_train)})', color='blue', density=True)
ax7.hist(max_probs_test, bins=30, alpha=0.6, label=f'Test (n={len(max_probs_test)})', color='orange', density=True)
ax7.set_title('Prediction Confidence', fontsize=12, fontweight='bold')
ax7.set_xlabel('Max Probability')
ax7.set_ylabel('Density')
ax7.legend()
ax7.grid(True, alpha=0.3)

ax8 = fig.add_subplot(gs[2, 1])
acc_per_class_train = []
acc_per_class_test = []
for i in range(n_classes):
    mask_train = train_labels == i
    mask_test = val_labels == i
    acc_per_class_train.append((train_preds[mask_train] == i).mean() if mask_train.sum() > 0 else 0)
    acc_per_class_test.append((val_preds[mask_test] == i).mean() if mask_test.sum() > 0 else 0)
x = np.arange(n_classes)
bars1 = ax8.bar(x - width/2, acc_per_class_train, width, label='Train', color='#2196F3', edgecolor='black', alpha=0.8)
bars2 = ax8.bar(x + width/2, acc_per_class_test, width, label='Test', color='#FF9800', edgecolor='black', alpha=0.8)
ax8.set_title('Per-Class Accuracy', fontsize=12, fontweight='bold')
ax8.set_xticks(x)
ax8.set_xticklabels(classes, rotation=45, ha='right')
ax8.set_ylabel('Accuracy')
ax8.set_ylim(0, 1.1)
ax8.legend()
ax8.grid(True, alpha=0.3, axis='y')

ax9 = fig.add_subplot(gs[2, 2])
ax9.axis('off')
table_data = [
    ['Metric', 'Train', 'Test', 'Diff'],
    ['Accuracy', f'{train_acc:.4f}', f'{test_acc:.4f}', f'{train_acc-test_acc:+.4f}'],
    ['Precision', f'{train_prec:.4f}', f'{test_prec:.4f}', f'{train_prec-test_prec:+.4f}'],
    ['Recall', f'{train_rec:.4f}', f'{test_rec:.4f}', f'{train_rec-test_rec:+.4f}'],
    ['F1', f'{train_f1:.4f}', f'{test_f1:.4f}', f'{train_f1-test_f1:+.4f}'],
    ['AUC-ROC', f'{train_auc_macro:.4f}', f'{test_auc_macro:.4f}', f'{train_auc_macro-test_auc_macro:+.4f}'],
    ['Kappa', f'{train_kappa:.4f}', f'{test_kappa:.4f}', f'{train_kappa-test_kappa:+.4f}'],
    ['MCC', f'{train_mcc:.4f}', f'{test_mcc:.4f}', f'{train_mcc-test_mcc:+.4f}'],
]
table = ax9.table(cellText=table_data[1:], colLabels=table_data[0], cellLoc='center', loc='center', colColours=['#f0f0f0']*4)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)
ax9.set_title('Metrics Summary', fontsize=12, fontweight='bold', pad=20)

ax10 = fig.add_subplot(gs[3, :])
overfit_score = train_acc - test_acc
if overfit_score > 0.05:
    status, color = 'OVERFITTING', 'red'
elif overfit_score < -0.05:
    status, color = 'UNDERFITTING', 'orange'
else:
    status, color = 'GOOD FIT', 'green'
ax10.text(0.5, 0.5, f'Overfitting Analysis\n\nTrain Accuracy: {train_acc*100:.2f}%\nTest Accuracy: {test_acc*100:.2f}%\n\nGap: {overfit_score*100:.2f}%\n\nStatus: {status}',
         transform=ax10.transAxes, fontsize=14, verticalalignment='center', horizontalalignment='center',
         bbox=dict(boxstyle='round', facecolor=color, alpha=0.3))
ax10.axis('off')

plt.suptitle('Classifier Evaluation Dashboard', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\n' + '=' * 70)
print('       CLASSIFICATION REPORT - TRAIN')
print('=' * 70)
print(classification_report(train_labels, train_preds, target_names=classes, digits=4))
print('\n' + '=' * 70)
print('       CLASSIFICATION REPORT - TEST')
print('=' * 70)
print(classification_report(val_labels, val_preds, target_names=classes, digits=4))

In [ ]:
#@title Cell 6: Prepare segmentation data (BraTS)
import os, glob, numpy as np

def _find_root(start):
    d = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(d, 'src', 'brain_tumor_dx')):
            return d
        parent = os.path.dirname(d)
        if parent == d: break
        d = parent
    return start

PROJECT_DIR = _find_root(os.getcwd())
os.chdir(PROJECT_DIR)
DATA_DIR = os.path.join(PROJECT_DIR, 'data')

SEG_DATA = None
search_dirs = [
    os.path.join(DATA_DIR, 'raw', 'segmentation'),
    os.path.join(DATA_DIR, 'segmentation'),
    os.path.join(DATA_DIR, 'raw'),
    DATA_DIR,
]

for d in search_dirs:
    if not os.path.isdir(d):
        continue
    if os.path.exists(os.path.join(d, 'case_001', 'image.nii.gz')):
        SEG_DATA = d
        print(f'Found NIfTI format: {d}')
        break
    raw_files = glob.glob(os.path.join(d, 'BraTS20*_flair.nii.gz'))
    if raw_files:
        SEG_DATA = d
        print(f'Found raw BraTS: {d}')
        break
    for sub in os.listdir(d):
        sub_path = os.path.join(d, sub)
        if not os.path.isdir(sub_path):
            continue
        if os.path.exists(os.path.join(sub_path, 'case_001', 'image.nii.gz')):
            SEG_DATA = sub_path
            print(f'Found NIfTI format: {sub_path}')
            break

if SEG_DATA:
    n_cases = len([x for x in os.listdir(SEG_DATA) if os.path.isdir(os.path.join(SEG_DATA, x))])
    print(f'Segmentation data ready: {n_cases} cases at {SEG_DATA}')
else:
    print('No segmentation data found.')
    print('Place data in data/raw/segmentation/case_001/{{image.nii.gz, mask.nii.gz}}')

In [ ]:
#@title Cell 7: Load Segmenter + Evaluate on Train/Test
import os, sys, torch, numpy as np
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
from scipy.spatial.distance import directed_hausdorff

def _find_root(start):
    d = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(d, 'src', 'brain_tumor_dx')):
            return d
        parent = os.path.dirname(d)
        if parent == d: break
        d = parent
    return start

PROJECT_DIR = _find_root(os.getcwd())
os.chdir(PROJECT_DIR)
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
SEG_DATA = os.path.join(DATA_DIR, 'raw', 'segmentation')
if not os.path.isdir(SEG_DATA) or not os.path.exists(os.path.join(SEG_DATA, 'case_001', 'image.nii.gz')):
    for d in [os.path.join(DATA_DIR, 'raw'), DATA_DIR]:
        if os.path.exists(os.path.join(d, 'case_001', 'image.nii.gz')):
            SEG_DATA = d
            break

SEG_DEVICE = 'cpu'

src_path = os.path.join(PROJECT_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from brain_tumor_dx.config import settings
from brain_tumor_dx.data.datasets import SegmentationDataset
from brain_tumor_dx.models.segmentation import TumorSegmenter

VAL_SPLIT = 0.2
full_dataset = SegmentationDataset(SEG_DATA)
n_val = int(len(full_dataset) * VAL_SPLIT)
n_train = len(full_dataset) - n_val

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_dataset, [n_train, n_val], generator=generator)

print(f'Total: {len(full_dataset)} cases')
print(f'Train: {n_train} | Test: {n_val}')

CKPT = os.path.join(CHECKPOINT_DIR, 'segmentation.pt')
if not os.path.exists(CKPT):
    raise FileNotFoundError(f'Checkpoint not found: {CKPT}')

model = TumorSegmenter().to(SEG_DEVICE)
model.load_state_dict(torch.load(CKPT, map_location=SEG_DEVICE))
model.eval()
print(f'Loaded checkpoint: {CKPT}')
print(f'Evaluating on CPU...')

def evaluate_segmenter(model, loader, device):
    results = []
    with torch.no_grad():
        for images, masks in tqdm(loader, desc='Evaluating', leave=False):
            images = images.to(device)
            masks = masks.to(device)
            pred = model(images)
            pred_bin = (torch.sigmoid(pred) > 0.5).float()
            # Resize mask if shapes differ
            if pred_bin.shape != masks.shape:
                masks = torch.nn.functional.interpolate(masks, size=pred_bin.shape[2:], mode='nearest')
            intersection = (pred_bin * masks).sum().item()
            dice = (2 * intersection + 1e-8) / (pred_bin.sum().item() + masks.sum().item() + 1e-8)
            union = pred_bin.sum().item() + masks.sum().item() - intersection
            iou = (intersection + 1e-8) / (union + 1e-8)
            v_pred = pred_bin.sum().item()
            v_true = masks.sum().item()
            vol_sim = 1.0 - abs(v_pred - v_true) / (v_pred + v_true + 1e-8)
            pred_np = pred_bin.squeeze().cpu().numpy()
            true_np = masks.squeeze().cpu().numpy()
            pred_coords = np.argwhere(pred_np > 0.5)
            true_coords = np.argwhere(true_np > 0.5)
            if len(pred_coords) > 0 and len(true_coords) > 0:
                h_dist = max(directed_hausdorff(pred_coords, true_coords)[0],
                           directed_hausdorff(true_coords, pred_coords)[0])
            else:
                h_dist = float('inf') if (len(pred_coords) > 0) != (len(true_coords) > 0) else 0.0
            results.append({'dice': dice, 'iou': iou, 'hausdorff': h_dist, 'vol_sim': vol_sim, 'v_pred': v_pred, 'v_true': v_true})
    return results

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)

print('\nEvaluating TRAIN set...')
train_results = evaluate_segmenter(model, train_loader, SEG_DEVICE)
print('Evaluating TEST set...')
val_results = evaluate_segmenter(model, val_loader, SEG_DEVICE)

train_dices = np.array([r['dice'] for r in train_results])
train_ious = np.array([r['iou'] for r in train_results])
train_hausdorff = np.array([r['hausdorff'] for r in train_results if r['hausdorff'] != float('inf')])
train_vol_sim = np.array([r['vol_sim'] for r in train_results])
test_dices = np.array([r['dice'] for r in val_results])
test_ious = np.array([r['iou'] for r in val_results])
test_hausdorff = np.array([r['hausdorff'] for r in val_results if r['hausdorff'] != float('inf')])
test_vol_sim = np.array([r['vol_sim'] for r in val_results])

print(f'\nTrain: {len(train_results)} cases | Test: {len(val_results)} cases')

In [ ]:
#@title Cell 8: Comprehensive SEGMENTER Evaluation (Train vs Test)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

print('=' * 70)
print('       SEGMENTER TRAINING SET METRICS')
print('=' * 70)
print(f'  Cases:         {len(train_results)}')
print(f'  Mean Dice:     {np.mean(train_dices):.4f} (+/- {np.std(train_dices):.4f})')
print(f'  Median Dice:   {np.median(train_dices):.4f}')
print(f'  Mean IoU:      {np.mean(train_ious):.4f} (+/- {np.std(train_ious):.4f})')
if len(train_hausdorff) > 0:
    print(f'  Mean Hausdorff: {np.mean(train_hausdorff):.2f} voxels')
print(f'  Mean Vol Sim:  {np.mean(train_vol_sim):.4f}')

print('\n' + '=' * 70)
print('       SEGMENTER TEST/VALIDATION SET METRICS')
print('=' * 70)
print(f'  Cases:         {len(val_results)}')
print(f'  Mean Dice:     {np.mean(test_dices):.4f} (+/- {np.std(test_dices):.4f})')
print(f'  Median Dice:   {np.median(test_dices):.4f}')
print(f'  Mean IoU:      {np.mean(test_ious):.4f} (+/- {np.std(test_ious):.4f})')
if len(test_hausdorff) > 0:
    print(f'  Mean Hausdorff: {np.mean(test_hausdorff):.2f} voxels')
print(f'  Mean Vol Sim:  {np.mean(test_vol_sim):.4f}')

fig = plt.figure(figsize=(20, 18))
gs = gridspec.GridSpec(4, 3, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(train_dices, bins=20, alpha=0.6, label=f'Train (n={len(train_dices)})', color='blue', density=True)
ax1.hist(test_dices, bins=20, alpha=0.6, label=f'Test (n={len(test_dices)})', color='orange', density=True)
ax1.axvline(np.mean(train_dices), color='blue', linestyle='--', lw=2, label=f'Train: {np.mean(train_dices):.3f}')
ax1.axvline(np.mean(test_dices), color='orange', linestyle='--', lw=2, label=f'Test: {np.mean(test_dices):.3f}')
ax1.set_title('Dice Distribution', fontsize=12, fontweight='bold')
ax1.set_xlabel('Dice Score')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(train_ious, bins=20, alpha=0.6, label=f'Train', color='blue', density=True)
ax2.hist(test_ious, bins=20, alpha=0.6, label=f'Test', color='orange', density=True)
ax2.axvline(np.mean(train_ious), color='blue', linestyle='--', lw=2, label=f'Train: {np.mean(train_ious):.3f}')
ax2.axvline(np.mean(test_ious), color='orange', linestyle='--', lw=2, label=f'Test: {np.mean(test_ious):.3f}')
ax2.set_title('IoU Distribution', fontsize=12, fontweight='bold')
ax2.set_xlabel('IoU Score')
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(gs[0, 2])
metrics_names = ['Dice', 'IoU', 'Vol Sim']
train_metrics = [np.mean(train_dices), np.mean(train_ious), np.mean(train_vol_sim)]
test_metrics = [np.mean(test_dices), np.mean(test_ious), np.mean(test_vol_sim)]
x = np.arange(len(metrics_names))
width = 0.35
bars1 = ax3.bar(x - width/2, train_metrics, width, label='Train', color='#2196F3', edgecolor='black', alpha=0.8)
bars2 = ax3.bar(x + width/2, test_metrics, width, label='Test', color='#FF9800', edgecolor='black', alpha=0.8)
ax3.set_title('Train vs Test Metrics', fontsize=12, fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(metrics_names)
ax3.set_ylim(0, 1.1)
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')
for bar in bars1:
    h = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., h + 0.02, f'{h:.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    h = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., h + 0.02, f'{h:.3f}', ha='center', va='bottom', fontsize=9)

ax4 = fig.add_subplot(gs[1, 0])
case_ids = range(1, len(train_dices) + 1)
bar_colors = ['green' if d >= 0.8 else 'orange' if d >= 0.5 else 'red' for d in train_dices]
ax4.bar(case_ids, train_dices, color=bar_colors, edgecolor='black', alpha=0.8)
ax4.axhline(y=0.8, color='green', linestyle='--', alpha=0.5, label='Good (0.8)')
ax4.axhline(y=0.5, color='orange', linestyle='--', alpha=0.5, label='Acceptable (0.5)')
ax4.set_title('Per-Case Dice (Train)', fontsize=12, fontweight='bold')
ax4.set_xlabel('Case')
ax4.set_ylabel('Dice')
ax4.set_ylim(0, 1.05)
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)

ax5 = fig.add_subplot(gs[1, 1])
case_ids = range(1, len(test_dices) + 1)
bar_colors = ['green' if d >= 0.8 else 'orange' if d >= 0.5 else 'red' for d in test_dices]
ax5.bar(case_ids, test_dices, color=bar_colors, edgecolor='black', alpha=0.8)
ax5.axhline(y=0.8, color='green', linestyle='--', alpha=0.5, label='Good (0.8)')
ax5.axhline(y=0.5, color='orange', linestyle='--', alpha=0.5, label='Acceptable (0.5)')
ax5.set_title('Per-Case Dice (Test)', fontsize=12, fontweight='bold')
ax5.set_xlabel('Case')
ax5.set_ylabel('Dice')
ax5.set_ylim(0, 1.05)
ax5.legend(fontsize=9)
ax5.grid(True, alpha=0.3)

ax6 = fig.add_subplot(gs[1, 2])
if len(train_hausdorff) > 0 and len(test_hausdorff) > 0:
    ax6.hist(train_hausdorff, bins=20, alpha=0.6, label=f'Train', color='blue', density=True)
    ax6.hist(test_hausdorff, bins=20, alpha=0.6, label=f'Test', color='orange', density=True)
    ax6.axvline(np.mean(train_hausdorff), color='blue', linestyle='--', lw=2, label=f'Train: {np.mean(train_hausdorff):.1f}')
    ax6.axvline(np.mean(test_hausdorff), color='orange', linestyle='--', lw=2, label=f'Test: {np.mean(test_hausdorff):.1f}')
ax6.set_title('Hausdorff Distance', fontsize=12, fontweight='bold')
ax6.set_xlabel('Distance (voxels)')
ax6.legend()
ax6.grid(True, alpha=0.3)

ax7 = fig.add_subplot(gs[2, 0])
ranges = ['<0.5', '0.5-0.7', '0.7-0.8', '0.8-0.9', '0.9-1.0']
train_counts = [np.sum(train_dices < 0.5), np.sum((train_dices >= 0.5) & (train_dices < 0.7)), np.sum((train_dices >= 0.7) & (train_dices < 0.8)), np.sum((train_dices >= 0.8) & (train_dices < 0.9)), np.sum(train_dices >= 0.9)]
test_counts = [np.sum(test_dices < 0.5), np.sum((test_dices >= 0.5) & (test_dices < 0.7)), np.sum((test_dices >= 0.7) & (test_dices < 0.8)), np.sum((test_dices >= 0.8) & (test_dices < 0.9)), np.sum(test_dices >= 0.9)]
x = np.arange(len(ranges))
bars1 = ax7.bar(x - width/2, train_counts, width, label='Train', color='#2196F3', edgecolor='black', alpha=0.8)
bars2 = ax7.bar(x + width/2, test_counts, width, label='Test', color='#FF9800', edgecolor='black', alpha=0.8)
ax7.set_title('Dice Distribution by Range', fontsize=12, fontweight='bold')
ax7.set_xticks(x)
ax7.set_xticklabels(ranges)
ax7.set_ylabel('Count')
ax7.legend()
ax7.grid(True, alpha=0.3, axis='y')

ax8 = fig.add_subplot(gs[2, 1])
train_v_pred = np.array([r['v_pred'] for r in train_results])
train_v_true = np.array([r['v_true'] for r in train_results])
test_v_pred = np.array([r['v_pred'] for r in val_results])
test_v_true = np.array([r['v_true'] for r in val_results])
ax8.scatter(train_v_true, train_v_pred, c='blue', alpha=0.6, label='Train', edgecolors='black')
ax8.scatter(test_v_true, test_v_pred, c='orange', alpha=0.6, label='Test', edgecolors='black')
max_vol = max(max(train_v_true.max(), train_v_pred.max()), max(test_v_true.max(), test_v_pred.max())) * 1.1
ax8.plot([0, max_vol], [0, max_vol], 'r--', lw=2, label='Perfect')
ax8.set_title('Pred vs True Volume', fontsize=12, fontweight='bold')
ax8.set_xlabel('True Volume')
ax8.set_ylabel('Pred Volume')
ax8.legend()
ax8.grid(True, alpha=0.3)

ax9 = fig.add_subplot(gs[2, 2])
ax9.axis('off')
table_data = [
    ['Metric', 'Train', 'Test', 'Diff'],
    ['Mean Dice', f'{np.mean(train_dices):.4f}', f'{np.mean(test_dices):.4f}', f'{np.mean(train_dices)-np.mean(test_dices):+.4f}'],
    ['Mean IoU', f'{np.mean(train_ious):.4f}', f'{np.mean(test_ious):.4f}', f'{np.mean(train_ious)-np.mean(test_ious):+.4f}'],
    ['Mean Vol Sim', f'{np.mean(train_vol_sim):.4f}', f'{np.mean(test_vol_sim):.4f}', f'{np.mean(train_vol_sim)-np.mean(test_vol_sim):+.4f}'],
]
table = ax9.table(cellText=table_data[1:], colLabels=table_data[0], cellLoc='center', loc='center', colColours=['#f0f0f0']*4)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)
ax9.set_title('Metrics Summary', fontsize=12, fontweight='bold', pad=20)

ax10 = fig.add_subplot(gs[3, :])
dice_gap = np.mean(train_dices) - np.mean(test_dices)
if dice_gap > 0.1:
    status, color = 'OVERFITTING', 'red'
elif dice_gap < -0.1:
    status, color = 'UNDERFITTING', 'orange'
else:
    status, color = 'GOOD FIT', 'green'
ax10.text(0.5, 0.5, f'Overfitting Analysis\n\nTrain Dice: {np.mean(train_dices)*100:.2f}%\nTest Dice: {np.mean(test_dices)*100:.2f}%\n\nGap: {dice_gap*100:.2f}%\n\nStatus: {status}',
         transform=ax10.transAxes, fontsize=14, verticalalignment='center', horizontalalignment='center',
         bbox=dict(boxstyle='round', facecolor=color, alpha=0.3))
ax10.axis('off')

plt.suptitle('Segmenter Evaluation Dashboard', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\n' + '=' * 70)
print('       PER-CASE RESULTS - TRAIN')
print('=' * 70)
print(f'{"Case":>6} {"Dice":>8} {"IoU":>8} {"Hausdorff":>10} {"Vol Sim":>8}')
print('-' * 44)
for i, r in enumerate(train_results):
    hd = f'{r["hausdorff"]:.1f}' if r['hausdorff'] != float('inf') else 'inf'
    print(f'{i+1:>6} {r["dice"]:>8.4f} {r["iou"]:>8.4f} {hd:>10} {r["vol_sim"]:>8.4f}')

print('\n' + '=' * 70)
print('       PER-CASE RESULTS - TEST')
print('=' * 70)
print(f'{"Case":>6} {"Dice":>8} {"IoU":>8} {"Hausdorff":>10} {"Vol Sim":>8}')
print('-' * 44)
for i, r in enumerate(val_results):
    hd = f'{r["hausdorff"]:.1f}' if r['hausdorff'] != float('inf') else 'inf'
    print(f'{i+1:>6} {r["dice"]:>8.4f} {r["iou"]:>8.4f} {hd:>10} {r["vol_sim"]:>8.4f}')